In [ ]:
import re
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

daftar_JudulSkripsi = [
    "PENGARUH SELF-EFFICACY TERHADAP SURFACE LEARNING MAHASISWA PRODI PENDIDIKAN TEKNIK INFORMATIKA DAN KOMPUTER UNIVERSITAS NEGERI MAKASSAR PADA ERA PESATNYA PENGGUNAAN ARTIFICIAL INTELLIGENCE",

    "PENGARUH LITERASI DIGITAL TERHADAP OPTIMALISASI PEMANFAATAN ARTIFICIAL INTELLIGENCE (AI) PADA PEMBELAJARAN MANDIRI MAHASISWA PENDIDIKAN TEKNIK INFORMATIKA DAN KOMPUTER FT UNM",

    "SISTEM PENENTUAN JENIS TANAMAN BERDASARKAN KARAKTERISTIK CUACA DAN PH TANAH BERBASIS ARTIFICIAL INTELLIGENCE",

    "PENGEMBANGAN ALAT PENDETEKSI PENAMPUNG AIR PENUH DENGAN MENGGUNAKAN SISTEM IOT TERINTEGRASI SMARTPHONE",

    "PERANCANGAN APLIKASI SMART GARAGE BERBASIS IOT UNTUK HUNIAN PERUMAHAN",

    "RANCANG BANGUN ALAT DETEKSI KEBOCORAN PIPA PDAM BERBASIS INTERNET OF THINGS (IOT)",

    "OPTIMASI KINERJA ROUTING DINAMIS MENGGUNAKAN ALGORITMA OPEN SHORTEST PATH FIRST (OSPF) DALAM TOPOLOGI MESH PADA JARINGAN LAN",

    "PENERAPAN ANTENA SEKTORAL UNTUK MEMPERLUAS JANGKAUAN JARINGAN WIFI DI FAKULTAS TEKNIK UNM",

    "SISTEM KEAMANAN JARINGAN TERHADAP SERANGAN DOS (DENIAL OF SERVICE) MENGGUNAKAN SNORT DAN FIREWALL BERBASIS LINUX OS",

    "PENGEMBANGAN KEBIJAKAN KEAMANAN INFORMASI UNTUK INFRASTRUKTUR JARINGAN KOMPUTER PT. PLN PERSERO UID SULSELRABAR BERBASIS ISO 27001"
]

def normalisasi(teks):
    teks = teks.lower()
    teks = teks.replace(
        "internet of things",
        "iot"
    )
    return re.sub(r"[^a-z0-9\s]", " ", teks)


judul_normal = [
    normalisasi(judul)
    for judul in daftar_JudulSkripsi
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(judul_normal)

ground_truth = {
    "Artificial Intelligence": {1, 2, 3},
    "Jaringan": {7, 8, 9, 10},
    "IoT": {4, 5, 6}
}

def evaluasi(query, relevan):
    skor = cosine_similarity(
        vectorizer.transform([normalisasi(query)]),
        X
    ).flatten()

    ranking = [
        nomor + 1
        for nomor in skor.argsort()[::-1]
        if skor[nomor] > 0
    ][:3]

    jumlah_hit = sum(
        dokumen in relevan
        for dokumen in ranking
    )

    precision = jumlah_hit / 3
    recall = jumlah_hit / len(relevan)

    f1 = (
        0
        if precision + recall == 0
        else 2 * precision * recall
        / (precision + recall)
    )

    hit = 0
    total_precision = 0

    for posisi, dokumen in enumerate(ranking, start=1):
        if dokumen in relevan:
            hit += 1
            total_precision += hit / posisi

    ap = total_precision / len(relevan)

    return ranking, precision, recall, f1, ap

hasil = []

for query, relevan in ground_truth.items():
    ranking, precision, recall, f1, ap = evaluasi(
        query,
        relevan
    )

    hasil.append({
        "Query": query,
        "Hasil Ranking": ranking,
        "Precision@3": precision,
        "Recall": recall,
        "F1": f1,
        "AP": ap
    })

df_hasil = pd.DataFrame(hasil)
map_score = df_hasil["AP"].mean()

print("=" * 60)
print("HASIL EVALUASI SISTEM TEMU KEMBALI INFORMASI")
print("=" * 60)

display(
    df_hasil.style.format({
        "Precision@3": "{:.3f}",
        "Recall": "{:.3f}",
        "F1": "{:.3f}",
        "AP": "{:.3f}"
    })
)

print("\n" + "=" * 15)
print("HASIL MAP")
print("=" * 15)
print(f"MAP = {map_score:.3f}")

df_ground_truth = pd.DataFrame([
    {
        "No": nomor,
        "Judul": judul,
        "Artificial Intelligence": (
            "✓"
            if nomor in ground_truth["Artificial Intelligence"]
            else "✗"
        ),
        "Jaringan": (
            "✓"
            if nomor in ground_truth["Jaringan"]
            else "✗"
        ),
        "IoT": (
            "✓"
            if nomor in ground_truth["IoT"]
            else "✗"
        )
    }
    for nomor, judul in enumerate(
        daftar_JudulSkripsi,
        start=1
    )
])

print("\n" + "=" * 15)
print("GROUND TRUTH")
print("=" * 15)

display(df_ground_truth)

terburuk = df_hasil.loc[
    df_hasil["AP"].idxmin()
]

print("\n" + "=" * 15)
print("ANALISIS")
print("=" * 15)

print(
    f"Kueri dengan performa paling buruk adalah "
    f"'{terburuk['Query']}'."
)

print(
    f"Nilai AP kueri tersebut adalah "
    f"{terburuk['AP']:.3f}."
)

print(
    "Hal ini menunjukkan bahwa tidak semua dokumen "
    "relevan berhasil ditemukan pada hasil teratas."
)

print(
    "Penyebabnya adalah TF-IDF menentukan relevansi "
    "berdasarkan kesamaan kata dalam judul."
)

print(
    f"Nilai MAP keseluruhan adalah "
    f"{map_score:.3f}."
)

HASIL EVALUASI SISTEM TEMU KEMBALI INFORMASI


,Query,Hasil Ranking,Precision@3,Recall,F1,AP
0,Artificial Intelligence,"[np.int64(3), np.int64(2), np.int64(1)]",1.000,1.000,1.000,1.000
1,Jaringan,"[np.int64(8), np.int64(9), np.int64(10)]",1.000,0.750,0.857,0.750
2,IoT,"[np.int64(6), np.int64(5), np.int64(4)]",1.000,1.000,1.000,1.000



HASIL MAP
MAP = 0.917

GROUND TRUTH


,No,Judul,Artificial Intelligence,Jaringan,IoT
0,1,PENGARUH SELF-EFFICACY TERHADAP SURFACE LEARNI...,✓,✗,✗
1,2,PENGARUH LITERASI DIGITAL TERHADAP OPTIMALISAS...,✓,✗,✗
2,3,SISTEM PENENTUAN JENIS TANAMAN BERDASARKAN KAR...,✓,✗,✗
3,4,PENGEMBANGAN ALAT PENDETEKSI PENAMPUNG AIR PEN...,✗,✗,✓
4,5,PERANCANGAN APLIKASI SMART GARAGE BERBASIS IOT...,✗,✗,✓
5,6,RANCANG BANGUN ALAT DETEKSI KEBOCORAN PIPA PDA...,✗,✗,✓
6,7,OPTIMASI KINERJA ROUTING DINAMIS MENGGUNAKAN A...,✗,✓,✗
7,8,PENERAPAN ANTENA SEKTORAL UNTUK MEMPERLUAS JAN...,✗,✓,✗
8,9,SISTEM KEAMANAN JARINGAN TERHADAP SERANGAN DOS...,✗,✓,✗
9,10,PENGEMBANGAN KEBIJAKAN KEAMANAN INFORMASI UNTU...,✗,✓,✗



ANALISIS
Kueri dengan performa paling buruk adalah 'Jaringan'.
Nilai AP kueri tersebut adalah 0.750.
Hal ini menunjukkan bahwa tidak semua dokumen relevan berhasil ditemukan pada hasil teratas.
Penyebabnya adalah TF-IDF menentukan relevansi berdasarkan kesamaan kata dalam judul.
Nilai MAP keseluruhan adalah 0.917.
